# Talaqah AI (طلاقة)
## A Multi-Agent Presentation, Voice, Video, and Public-Speaking Coach

**SDAIA Academy — Building Agentic AI Systems Capstone Project**  
**Trainer:** Eng. Mohammed Albeladi  
**Track:** A — Supervisor + Specialist Workers  
**Workflow pattern:** Evaluator–Optimizer  
**Cohort:** August 2026

This notebook is the complete, executable capstone artifact. It supports real transcript, PowerPoint/PDF, microphone audio, uploaded audio, webcam video, and uploaded video inputs. Keep its outputs when submitting the project.


## Architecture and rubric coverage

| Capstone area | Talaqah implementation and visible evidence |
|---|---|
| Agent fundamentals | Three specialist agents call real RAG, transcript, slide, audio, video, and deck-generation tools; tool calls and arguments are printed. |
| Multi-agent systems | A structured-output LLM supervisor returns `RoutePlan(destinations, reason, confidence)` and can select several workers. Selected specialists run with Functional API fan-out/fan-in. |
| RAG | Knowledge ZIP → safe extraction → document loading → chunking → Sentence Transformer embeddings → FAISS → semantic retrieval with source filenames and scores. |
| Context and state | `InMemorySaver` provides thread state; `InMemoryStore` keeps learner facts across threads without unbounded message history. |
| Human-in-the-loop | A real `interrupt()` pauses before publication; approval continues with `Command(resume=...)`. |
| Functional API | `@task`, `@entrypoint`, parallel task futures, retry policies, rate limiting, user-fixable input interrupt, and explicit validation. |
| Workflow pattern | Evaluator–Optimizer: draft → structured evaluation → optimized report → final evaluation. |
| Observability | LangSmith tracing uses the course variable plus current variables; the notebook inspects the real traced run. |
| Audio/video extension | FFmpeg/Pydub measure duration, pauses, silence, and signal variation; Gemini analyzes real audio/video qualitatively. Audio is processed even with a supplied transcript. |

### Execution flow

1. The supervisor reads the request and available input types and selects every relevant specialist.
2. Content & Slides, Delivery, and Confidence workers execute in parallel when the full-team option is enabled.
3. Each worker retrieves grounded guidance and calls the tools appropriate to the real files.
4. A report task merges all worker evidence.
5. The Evaluator–Optimizer loop reviews and improves the report.
6. Human approval is required before the approved JSON and Markdown artifacts are published.


## 0. Install dependencies

The project uses the LangGraph **Functional API**, LangChain agents, Gemini, LangSmith, FAISS, Sentence Transformers, PDF/DOCX/PPTX readers, and Pydantic.


In [1]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install "google-auth==2.49.0" "langchain>=1.1" "langgraph>=1.2" langchain-google-genai langsmith langchain-text-splitters pydantic pypdf python-docx python-pptx sentence-transformers faiss-cpu "gradio>=5,<7" google-genai pydub


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.7 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import uuid
import time
import shutil
import zipfile
import tempfile
import subprocess
from pathlib import Path
from typing import Any, Literal

import faiss
import numpy as np
from pydantic import BaseModel, Field
from pypdf import PdfReader
from docx import Document
from pptx import Presentation
from pydub import AudioSegment, silence
from sentence_transformers import SentenceTransformer

from google.colab import files, userdata
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command, RetryPolicy, interrupt
from langsmith import Client
from langchain_core.tracers.langchain import wait_for_all_tracers

print("✅ Imports completed, including audio/video dependencies.")


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


✅ Imports completed, including audio/video dependencies.


## 1. Secrets and LangSmith observability

In **Colab → Secrets**, add:

- `GEMINI_API_KEY`
- `LANGSMITH_API_KEY`

Do not paste either key into a code cell. The course rubric explicitly checks `LANGCHAIN_TRACING_V2`, so this notebook configures that exact variable and the current LangSmith variable as well.


In [3]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets, enable notebook access, then rerun this cell.")
if not LANGSMITH_API_KEY:
    raise ValueError("Add LANGSMITH_API_KEY to Colab Secrets, enable notebook access, then rerun this cell.")

LANGSMITH_PROJECT = "Talaqah-AI-Capstone-August-2026"

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"

langsmith_client = Client()
print("✅ Gemini and LangSmith secrets verified.")
print("Tracing project:", LANGSMITH_PROJECT)


✅ Gemini and LangSmith secrets verified.
Tracing project: Talaqah-AI-Capstone-August-2026


## 2. Upload the project inputs

Upload these two required files together:

1. `Talaqah_Knowledge_Base.zip`
2. `rehearsal_transcript.txt`

You may also upload a `.pptx` or presentation `.pdf`. If no presentation is uploaded, the notebook creates a small demo deck so slide-analysis tool calls remain reproducible.


In [4]:
INPUT_DIR = Path("/content/talaqah_inputs")
KNOWLEDGE_DIR = Path("/content/talaqah_knowledge_base")
OUTPUT_DIR = Path("/content/talaqah_outputs")

# A clean scoped folder prevents stale files from an earlier Colab run being selected accidentally.
for directory in (INPUT_DIR, KNOWLEDGE_DIR, OUTPUT_DIR):
    shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True, exist_ok=True)

print("Upload the knowledge ZIP. You may also select a transcript, presentation, audio, and/or video in the same dialog.")
uploaded = files.upload()
for filename, data in uploaded.items():
    (INPUT_DIR / Path(filename).name).write_bytes(data)

zip_files = sorted(INPUT_DIR.glob("*.zip"))
transcript_files = sorted(INPUT_DIR.glob("*.txt"))
presentation_files = sorted(INPUT_DIR.glob("*.pptx")) + sorted(INPUT_DIR.glob("*.pdf"))
audio_files = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in {".wav", ".mp3", ".m4a", ".aac", ".ogg", ".flac"}]
video_files = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in {".mp4", ".mov", ".mkv", ".avi", ".webm"}]

if not zip_files:
    raise FileNotFoundError("Upload SpeakUp_Knowledge_Base.zip (the knowledge-base ZIP supplied with the project).")

knowledge_zip = zip_files[0]
with zipfile.ZipFile(knowledge_zip) as archive:
    root = KNOWLEDGE_DIR.resolve()
    for member in archive.infolist():
        destination = (KNOWLEDGE_DIR / member.filename).resolve()
        if root not in destination.parents and destination != root:
            raise ValueError(f"Unsafe ZIP path rejected: {member.filename}")
    archive.extractall(KNOWLEDGE_DIR)

if transcript_files:
    transcript_path = transcript_files[0]
else:
    transcript_path = INPUT_DIR / "talaqah_demo_transcript.txt"
    transcript_path.write_text(
        "Hello everyone. Today I will explain how deliberate rehearsal builds confidence. "
        "First, practice the opening. Next, measure your pace and reduce filler words. "
        "Finally, repeat the presentation and use the feedback to improve.",
        encoding="utf-8",
    )
    print("ℹ️ No transcript was uploaded; a clearly labelled demo transcript was created for the notebook evidence cells.")

audio_path = audio_files[0] if audio_files else None
video_path = video_files[0] if video_files else None

print("✅ Knowledge ZIP extracted safely.")
print("Transcript:", transcript_path.name)
print("Optional audio:", audio_path.name if audio_path else "not supplied")
print("Optional video:", video_path.name if video_path else "not supplied")


Upload the knowledge ZIP. You may also select a transcript, presentation, audio, and/or video in the same dialog.


Saving rehearsal_transcript.txt to rehearsal_transcript.txt
Saving SpeakUp_Knowledge_Base.zip to SpeakUp_Knowledge_Base.zip
✅ Knowledge ZIP extracted safely.
Transcript: rehearsal_transcript.txt
Optional audio: not supplied
Optional video: not supplied


In [5]:
def create_demo_presentation(path: Path) -> Path:
    """Create a real PPTX for reproducible slide-analysis evidence."""
    prs = Presentation()
    title_slide = prs.slides.add_slide(prs.slide_layouts[0])
    title_slide.shapes.title.text = "Talaqah AI (طلاقة)"
    title_slide.placeholders[1].text = "A Multi-Agent Coach for Confident Presentations"

    slides = [
        ("The Problem", ["Presentation anxiety", "Overloaded slides", "Limited rehearsal feedback"]),
        ("The Solution", ["Specialist AI coaches", "Grounded RAG guidance", "Human-approved feedback"]),
        ("Key Takeaway", ["Practice deliberately", "Measure real progress", "Present with clarity and confidence"]),
    ]
    for title, bullets in slides:
        slide = prs.slides.add_slide(prs.slide_layouts[1])
        slide.shapes.title.text = title
        frame = slide.placeholders[1].text_frame
        frame.clear()
        for index, bullet in enumerate(bullets):
            paragraph = frame.paragraphs[0] if index == 0 else frame.add_paragraph()
            paragraph.text = bullet
    prs.save(path)
    return path

if presentation_files:
    presentation_path = presentation_files[0]
else:
    presentation_path = create_demo_presentation(INPUT_DIR / "talaqah_demo_presentation.pptx")

print("Presentation used for notebook evidence:", presentation_path.name)


Presentation used for notebook evidence: talaqah_demo_presentation.pptx


## 3. Complete RAG pipeline: load → split → embed → store → retrieve

**Architecture choice — Hybrid RAG:** indexing and semantic search are deterministic, while the supervisor and workers use LLM decisions to choose the relevant specialist and tool query. This fits Talaqah AI (طلاقة) because every coaching claim should be grounded in the curated presentation guides, while the path and retrieval query must adapt to the learner's request.


In [6]:
def read_pdf(path: Path) -> str:
    return "\n".join((page.extract_text() or "") for page in PdfReader(str(path)).pages)

def read_docx(path: Path) -> str:
    document = Document(str(path))
    return "\n".join(paragraph.text for paragraph in document.paragraphs)

def read_txt(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def load_documents(folder: Path) -> list[dict[str, str]]:
    loaded = []
    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue
        suffix = path.suffix.lower()
        if suffix == ".pdf":
            text = read_pdf(path)
        elif suffix == ".docx":
            text = read_docx(path)
        elif suffix == ".txt":
            text = read_txt(path)
        else:
            continue
        text = re.sub(r"[ \t]+", " ", text).strip()
        if text:
            loaded.append({"source": path.name, "text": text})
    return loaded

documents = load_documents(KNOWLEDGE_DIR)
if len(documents) < 2:
    raise RuntimeError("The knowledge base did not load correctly.")

print(f"✅ Loaded {len(documents)} documents.")
for document in documents:
    print(f"- {document['source']}: {len(document['text'])} characters")

✅ Loaded 7 documents.
- body_language_guide.pdf: 931 characters
- confidence_and_anxiety.pdf: 1300 characters
- presentation_rubric.pdf: 1627 characters
- presentation_structure.pdf: 1066 characters
- public_speaking_guide.pdf: 1501 characters
- slide_design_guide.pdf: 1240 characters
- voice_delivery_guide.pdf: 1116 characters


In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for document in documents:
    for chunk_id, chunk_text in enumerate(splitter.split_text(document["text"])):
        chunks.append({
            "source": document["source"],
            "chunk_id": chunk_id,
            "text": chunk_text,
        })

if len(chunks) <= len(documents):
    raise RuntimeError("Splitting evidence failed: chunk count must exceed document count.")

print(f"✅ Genuine splitting confirmed: {len(documents)} documents → {len(chunks)} chunks.")

✅ Genuine splitting confirmed: 7 documents → 23 chunks.


In [8]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.asarray(embeddings, dtype="float32")

vector_index = faiss.IndexFlatIP(embeddings.shape[1])
vector_index.add(embeddings)

print("✅ FAISS vector store ready.")
print("Stored vectors:", vector_index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS vector store ready.
Stored vectors: 23


## 4. Real tools

These functions are not hardcoded f-string “tools.” Each one performs real work with its arguments: vector retrieval, file parsing, measured speaking-rate analysis, PowerPoint creation, or final report publication.


In [9]:
@tool
def retrieve_guidelines(query: str, k: int = 4) -> str:
    """Semantically retrieve presentation-coaching guidance from the FAISS knowledge base."""
    k = max(1, min(int(k), len(chunks)))
    query_vector = embedding_model.encode([query], normalize_embeddings=True)
    query_vector = np.asarray(query_vector, dtype="float32")
    scores, indices = vector_index.search(query_vector, k)
    results = []
    for score, index_value in zip(scores[0], indices[0]):
        if index_value < 0:
            continue
        item = chunks[int(index_value)]
        results.append({
            "source": item["source"],
            "chunk_id": item["chunk_id"],
            "score": round(float(score), 4),
            "text": item["text"],
        })
    return json.dumps({"query": query, "results": results}, ensure_ascii=False)


FILLER_PATTERNS = [
    "um", "uh", "you know", "basically", "actually",
    "اممم", "ا مم", "يعني", "طيب", "اه", "آه",
]


def _transcript_metrics(text: str, duration_minutes: float) -> dict:
    words = re.findall(r"\b\w+\b", text.lower(), flags=re.UNICODE)
    filler_counts = {}
    for filler in FILLER_PATTERNS:
        count = len(re.findall(rf"(?<!\w){re.escape(filler)}(?!\w)", text.lower()))
        if count:
            filler_counts[filler] = count
    word_count = len(words)
    wpm = word_count / duration_minutes if duration_minutes and duration_minutes > 0 else None
    pace = "not_calculated"
    if wpm is not None:
        pace = "slow" if wpm < 100 else "fast" if wpm > 160 else "controlled"
    sentences = [x.strip() for x in re.split(r"[.!?؟]+", text) if x.strip()]
    return {
        "word_count": word_count,
        "actual_duration_minutes": round(float(duration_minutes), 3) if duration_minutes else None,
        "actual_words_per_minute": round(wpm, 1) if wpm is not None else None,
        "pace_label": pace,
        "filler_words": filler_counts,
        "filler_total": sum(filler_counts.values()),
        "average_sentence_words": round(word_count / max(1, len(sentences)), 1),
    }


def _probe_media_duration_minutes(path_value: str) -> float | None:
    if not path_value or not Path(path_value).exists():
        return None
    command = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(path_value),
    ]
    try:
        value = float(subprocess.check_output(command, text=True).strip()) / 60.0
        return value if value > 0 else None
    except Exception:
        return None


def _audio_signal_metrics(path_value: str) -> dict:
    audio = AudioSegment.from_file(path_value)
    duration_seconds = len(audio) / 1000.0
    base_dbfs = None if audio.dBFS == float("-inf") else round(float(audio.dBFS), 2)
    threshold = -50 if base_dbfs is None else max(-50, base_dbfs - 16)
    silent_ranges = silence.detect_silence(
        audio,
        min_silence_len=600,
        silence_thresh=threshold,
    )
    pause_seconds = [round((end - start) / 1000.0, 2) for start, end in silent_ranges]
    chunk_levels = []
    for start in range(0, len(audio), 500):
        level = audio[start:start + 500].dBFS
        if level != float("-inf"):
            chunk_levels.append(float(level))
    total_pause = sum(pause_seconds)
    return {
        "duration_seconds": round(duration_seconds, 2),
        "average_loudness_dbfs": base_dbfs,
        "pause_count_over_0_6s": len(pause_seconds),
        "total_pause_seconds": round(total_pause, 2),
        "silence_ratio": round(total_pause / duration_seconds, 3) if duration_seconds else 0,
        "loudness_variation_db": round(float(np.std(chunk_levels)), 2) if chunk_levels else None,
        "measurement_note": "Signal measurements are deterministic; qualitative delivery observations are produced separately by Gemini.",
    }


_MEDIA_CLIENT = None


def _media_client():
    global _MEDIA_CLIENT
    if _MEDIA_CLIENT is None:
        from google import genai
        _MEDIA_CLIENT = genai.Client(api_key=GEMINI_API_KEY)
    return _MEDIA_CLIENT


def _parse_json_response(text: str) -> dict:
    cleaned = (text or "").strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.I)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        value = json.loads(cleaned)
        return value if isinstance(value, dict) else {"result": value}
    except Exception:
        return {"raw_observation": cleaned}


def _gemini_media_analysis(path_value: str, media_kind: str) -> dict:
    """Upload real media to Gemini and return grounded JSON observations with quota-aware retries."""
    client = _media_client()
    uploaded_file = client.files.upload(file=str(path_value))
    try:
        for _ in range(60):
            current = client.files.get(name=uploaded_file.name)
            state = str(getattr(getattr(current, "state", None), "name", getattr(current, "state", ""))).upper()
            if "FAILED" in state:
                raise RuntimeError("Gemini failed to process the uploaded media file.")
            if "PROCESSING" not in state:
                uploaded_file = current
                break
            time.sleep(2)
        else:
            raise TimeoutError("Gemini media processing did not finish within two minutes.")

        if media_kind == "audio":
            prompt = (
                "Analyze this real rehearsal audio. Return JSON only with keys: transcript, language, "
                "delivery_observations, pronunciation_observations, vocal_variety_observations, "
                "notable_pauses, and limitations. Do not invent precise numeric measurements."
            )
        else:
            prompt = (
                "Analyze this real rehearsal video including its audio. Return JSON only with keys: transcript, "
                "language, delivery_observations, visible_delivery_observations, slide_or_scene_observations, "
                "notable_pauses, and limitations. Describe visible behavior only; do not infer emotion, health, "
                "identity, personality, or other sensitive traits."
            )

        last_error = None
        for attempt in range(3):
            try:
                limiter = globals().get("rate_limiter")
                if limiter is not None:
                    try:
                        limiter.acquire(blocking=True)
                    except Exception:
                        pass
                response = client.models.generate_content(
                    model="gemini-3.5-flash-lite",
                    contents=[uploaded_file, prompt],
                    config={"response_mime_type": "application/json"},
                )
                return _parse_json_response(getattr(response, "text", ""))
            except Exception as exc:
                last_error = exc
                transient = any(token in str(exc).upper() for token in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
                if not transient or attempt == 2:
                    raise
                time.sleep(20 * (attempt + 1))
        raise last_error
    finally:
        try:
            client.files.delete(name=uploaded_file.name)
        except Exception:
            pass


def _read_optional_transcript(transcript_path: str | None) -> str:
    if transcript_path and Path(transcript_path).exists():
        return Path(transcript_path).read_text(encoding="utf-8", errors="ignore")
    return ""


@tool
def analyze_transcript_file(transcript_path: str, actual_duration_minutes: float) -> str:
    """Read a real transcript and calculate fillers and actual words per minute."""
    path = Path(transcript_path)
    if not path.exists() or path.suffix.lower() != ".txt":
        raise FileNotFoundError(f"Transcript not found: {transcript_path}")
    if actual_duration_minutes is None or float(actual_duration_minutes) <= 0:
        raise ValueError("actual_duration_minutes must be greater than zero for transcript-only pace analysis.")
    metrics = _transcript_metrics(path.read_text(encoding="utf-8", errors="ignore"), float(actual_duration_minutes))
    metrics["file"] = path.name
    return json.dumps(metrics, ensure_ascii=False)


@tool
def analyze_audio_file(audio_path: str, transcript_path: str = "", actual_duration_minutes: float = 0) -> str:
    """Analyze a real audio rehearsal: measured signal/pause metrics plus Gemini speech and delivery observations."""
    path = Path(audio_path)
    if not path.exists():
        raise FileNotFoundError(f"Audio not found: {audio_path}")
    probed_duration = _probe_media_duration_minutes(str(path))
    measured_duration = probed_duration or (float(actual_duration_minutes) if actual_duration_minutes else None)
    signal_metrics = _audio_signal_metrics(str(path))
    qualitative = _gemini_media_analysis(str(path), "audio")
    supplied_text = _read_optional_transcript(transcript_path)
    generated_text = str(qualitative.get("transcript", "") or "")
    metric_text = supplied_text or generated_text
    return json.dumps({
        "file": path.name,
        "input_type": "audio",
        "duration_source": "measured_from_media" if probed_duration else "user_supplied",
        "signal_metrics": signal_metrics,
        "transcript_metrics": _transcript_metrics(metric_text, measured_duration) if metric_text and measured_duration else None,
        "transcript_source": "uploaded_transcript" if supplied_text else "generated_from_audio",
        "generated_transcript": generated_text,
        "qualitative_audio_observations": qualitative,
    }, ensure_ascii=False)


@tool
def analyze_video_file(video_path: str, transcript_path: str = "", actual_duration_minutes: float = 0) -> str:
    """Analyze a real rehearsal video: visual delivery, speech, and measured audio-signal metrics."""
    path = Path(video_path)
    if not path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")
    probed_duration = _probe_media_duration_minutes(str(path))
    measured_duration = probed_duration or (float(actual_duration_minutes) if actual_duration_minutes else None)
    temporary_audio = Path(tempfile.gettempdir()) / f"talaqah_{uuid.uuid4().hex}.wav"
    signal_metrics = None
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-i", str(path), "-vn", "-ac", "1", "-ar", "16000", str(temporary_audio)],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        signal_metrics = _audio_signal_metrics(str(temporary_audio))
    except Exception as exc:
        signal_metrics = {"warning": f"Audio track could not be measured: {type(exc).__name__}"}
    finally:
        temporary_audio.unlink(missing_ok=True)

    qualitative = _gemini_media_analysis(str(path), "video")
    supplied_text = _read_optional_transcript(transcript_path)
    generated_text = str(qualitative.get("transcript", "") or "")
    metric_text = supplied_text or generated_text
    return json.dumps({
        "file": path.name,
        "input_type": "video",
        "duration_source": "measured_from_media" if probed_duration else "user_supplied",
        "signal_metrics": signal_metrics,
        "transcript_metrics": _transcript_metrics(metric_text, measured_duration) if metric_text and measured_duration else None,
        "transcript_source": "uploaded_transcript" if supplied_text else "generated_from_video",
        "generated_transcript": generated_text,
        "qualitative_video_observations": qualitative,
        "visual_safety_note": "Only observable presentation behavior is described; no sensitive-trait or emotion inference is performed.",
    }, ensure_ascii=False)


@tool
def analyze_slide_deck(presentation_path: str) -> str:
    """Read a real PPTX or PDF deck and detect dense, empty, or untitled slides."""
    path = Path(presentation_path)
    if not path.exists():
        raise FileNotFoundError(f"Presentation not found: {presentation_path}")
    slide_texts = []
    if path.suffix.lower() == ".pptx":
        prs = Presentation(str(path))
        for slide in prs.slides:
            slide_texts.append([shape.text.strip() for shape in slide.shapes if hasattr(shape, "text") and shape.text.strip()])
    elif path.suffix.lower() == ".pdf":
        slide_texts = [[page.extract_text() or ""] for page in PdfReader(str(path)).pages]
    else:
        raise ValueError("Only PPTX and PDF presentations are supported.")
    details = []
    for number, texts in enumerate(slide_texts, start=1):
        combined = " ".join(texts).strip()
        word_count = len(re.findall(r"\b\w+\b", combined, flags=re.UNICODE))
        title = texts[0] if texts else ""
        details.append({
            "slide": number,
            "title": title[:100],
            "word_count": word_count,
            "is_overloaded": word_count > 40,
            "is_empty": word_count == 0,
            "possibly_missing_title": not bool(title),
        })
    return json.dumps({
        "file": path.name,
        "slide_count": len(details),
        "overloaded_slides": [x["slide"] for x in details if x["is_overloaded"]],
        "empty_slides": [x["slide"] for x in details if x["is_empty"]],
        "slides": details,
    }, ensure_ascii=False)


@tool
def create_presentation_deck(title: str, slides_json: str, output_filename: str = "talaqah_generated_deck.pptx") -> str:
    """Create a real PowerPoint deck from a JSON list of slide titles and bullets."""
    slides = json.loads(slides_json)
    if not isinstance(slides, list) or not slides:
        raise ValueError("slides_json must be a non-empty JSON list.")
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", output_filename)
    if not safe_name.lower().endswith(".pptx"):
        safe_name += ".pptx"
    output_path = OUTPUT_DIR / safe_name
    prs = Presentation()
    first = prs.slides.add_slide(prs.slide_layouts[0])
    first.shapes.title.text = title
    first.placeholders[1].text = "Created by Talaqah AI"
    for spec in slides:
        slide = prs.slides.add_slide(prs.slide_layouts[1])
        slide.shapes.title.text = str(spec.get("title", "Untitled"))
        frame = slide.placeholders[1].text_frame
        frame.clear()
        for i, bullet in enumerate(spec.get("bullets", [])):
            paragraph = frame.paragraphs[0] if i == 0 else frame.add_paragraph()
            paragraph.text = str(bullet)
    prs.save(output_path)
    return json.dumps({"created_file": str(output_path), "slide_count": len(prs.slides)}, ensure_ascii=False)


print("✅ RAG, transcript, slide, audio, and video tools registered.")


✅ RAG, transcript, slide, audio, and video tools registered.


## 5. RAG retrieval proof

The question below has an answer written directly in the knowledge base. The output must show relevant source names and text; an empty or irrelevant result means the pipeline is not ready.


In [10]:
rag_test_raw = retrieve_guidelines.invoke({
    "query": "What should a strong presentation conclusion do?",
    "k": 4,
})
rag_test = json.loads(rag_test_raw)

print("Question:", rag_test["query"])
for number, item in enumerate(rag_test["results"], start=1):
    print(f"\nResult {number}: {item['source']} | score={item['score']}")
    print(item["text"][:350])

assert rag_test["results"], "Retriever returned no results."
assert any("conclusion" in item["text"].lower() for item in rag_test["results"]), "Expected conclusion guidance was not retrieved."
print("\n✅ RAG retrieval test passed.")

Question: What should a strong presentation conclusion do?

Result 1: presentation_structure.pdf | score=0.5587
findings, or argument.
Main Content
Organize the content into a small number of meaningful sections. Each section should have a
clear purpose and connect to the main message.
Transitions
Use transitions to show how one idea connects to the next. Avoid abrupt jumps between
unrelated topics.
Conclusion
Restate the main message, summarize the most imp

Result 2: public_speaking_guide.pdf | score=0.5538
Prefer short sentences and one main idea at a time.
Audience Awareness
Consider the audience's background, expectations, and knowledge level. Connect examples
to their interests and avoid unnecessary detail.
Opening
A strong opening can use a question, short story, surprising fact, problem statement, or clear
objective. The opening should quickly e

Result 3: public_speaking_guide.pdf | score=0.5532
point should support that message.
Transitions
Use explicit transitions between se

## 6. Structured outputs and the LLM supervisor

Every result that is consumed by code is validated with Pydantic. Routing is decided by the LLM, not by keyword matching.


In [11]:
WorkerName = Literal["content_slide_agent", "delivery_agent", "confidence_agent"]


class RoutePlan(BaseModel):
    destinations: list[WorkerName] = Field(min_length=1, max_length=3)
    reason: str = Field(description="Why these specialists are needed")
    confidence: float = Field(ge=0.0, le=1.0)


class ReportMetric(BaseModel):
    name: str
    value: str


class CoachingReport(BaseModel):
    overall_assessment: str
    strengths: list[str]
    areas_for_improvement: list[str]
    practice_actions: list[str]
    metrics: list[ReportMetric] = Field(default_factory=list)
    sources: list[str] = Field(default_factory=list)
    confidence_score: float = Field(ge=0.0, le=1.0)


class ReportEvaluation(BaseModel):
    score: int = Field(ge=1, le=5)
    grounded_in_evidence: bool
    specific_and_actionable: bool
    issues: list[str] = Field(default_factory=list)
    should_revise: bool


from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=1 / 16,
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    google_api_key=GEMINI_API_KEY,
    max_retries=0,
    rate_limiter=rate_limiter,
)

router_llm = llm.with_structured_output(RoutePlan, method="json_schema")
report_llm = llm.with_structured_output(CoachingReport, method="json_schema")
evaluator_llm = llm.with_structured_output(ReportEvaluation, method="json_schema")
optimizer_llm = llm.with_structured_output(CoachingReport, method="json_schema")

print("✅ Gemini and Pydantic structured-output models are ready.")


✅ Gemini and Pydantic structured-output models are ready.


## 7. Specialist worker agents

- **Content & Slide Agent:** analyzes or creates presentations.
- **Delivery Agent:** measures the rehearsal using the real recording duration.
- **Confidence Agent:** creates evidence-grounded anxiety and practice guidance.

Each agent has a different tool set. The printed `tool_calls` later are direct evidence that the model chose and executed tools.


In [12]:
CONTENT_SLIDE_TOOLS = [retrieve_guidelines, analyze_slide_deck, create_presentation_deck]
DELIVERY_TOOLS = [retrieve_guidelines, analyze_transcript_file, analyze_audio_file, analyze_video_file]
CONFIDENCE_TOOLS = [retrieve_guidelines]

content_slide_agent = create_agent(
    model=llm,
    tools=CONTENT_SLIDE_TOOLS,
    name="content_slide_agent",
    system_prompt=(
        "You are Talaqah's content and slide specialist. Always call retrieve_guidelines first. "
        "If a presentation path is supplied, you MUST call analyze_slide_deck. If asked to create a deck, "
        "call create_presentation_deck. Base claims on tool evidence and name source files."
    ),
)

delivery_agent = create_agent(
    model=llm,
    tools=DELIVERY_TOOLS,
    name="delivery_agent",
    system_prompt=(
        "You are Talaqah's delivery specialist. Always call retrieve_guidelines. "
        "If audio_path is supplied, you MUST call analyze_audio_file even when a transcript is also supplied. "
        "If video_path is supplied, you MUST call analyze_video_file even when a transcript is also supplied. "
        "If only a transcript is supplied, call analyze_transcript_file with the actual duration. "
        "When audio and video are both supplied, analyze both and compare them only as observable evidence. "
        "Copy measured values exactly and keep qualitative Gemini observations separate from deterministic metrics. "
        "Never infer emotions, identity, health, or personality from audio/video."
    ),
)

confidence_agent = create_agent(
    model=llm,
    tools=CONFIDENCE_TOOLS,
    name="confidence_agent",
    system_prompt=(
        "You are Talaqah's confidence specialist. Always call retrieve_guidelines. "
        "Give gradual practical rehearsal exercises; do not diagnose mental-health conditions and do not infer "
        "confidence from appearance. Use the user's stated goals and retrieved guidance."
    ),
)

WORKERS = {
    "content_slide_agent": content_slide_agent,
    "delivery_agent": delivery_agent,
    "confidence_agent": confidence_agent,
}

print("✅ Track A supervisor workers ready:", list(WORKERS))


✅ Track A supervisor workers ready: ['content_slide_agent', 'delivery_agent', 'confidence_agent']


## 8. LangGraph Functional API tasks and reliability

Implemented error strategies:

1. **Transient:** every LLM task uses a real `RetryPolicy(max_attempts=3)`.
2. **User-fixable:** missing transcript path or duration triggers `interrupt()` and accepts corrected input on resume.
3. **LLM-recoverable:** evaluator issues are passed back to the optimizer for revision.
4. **Unexpected:** there is no broad catch-all around the main workflow, so unexpected failures surface honestly.


In [13]:
def _should_retry_talaqah(exc: Exception) -> bool:
    """Retry transient provider failures and structured-output parse/validation failures, not bad user files."""
    text = f"{type(exc).__name__}: {exc}".upper()
    return any(token in text for token in (
        "429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "TIMEOUT",
        "CONNECTION", "CHATGOOGLEGENERATIVEAIERROR", "JSON", "VALIDATION",
    ))


TRANSIENT_RETRY = RetryPolicy(
    max_attempts=3,
    initial_interval=15.0,
    backoff_factor=2.0,
    max_interval=60.0,
    retry_on=_should_retry_talaqah,
)


def message_to_text(message: Any) -> str:
    content = getattr(message, "content", "")
    return content if isinstance(content, str) else json.dumps(content, ensure_ascii=False, default=str)


def compact_agent_result(worker_name: str, result: dict) -> dict:
    tool_calls, tool_outputs = [], []
    final_answer = ""
    for message in result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            tool_calls.append({"name": call.get("name"), "args": call.get("args", {})})
        if isinstance(message, ToolMessage):
            tool_outputs.append({
                "tool": getattr(message, "name", "tool"),
                "output": message_to_text(message)[:10000],
            })
        if isinstance(message, AIMessage) and not (getattr(message, "tool_calls", []) or []):
            final_answer = message_to_text(message)
    return {"worker": worker_name, "answer": final_answer, "tool_calls": tool_calls, "tool_outputs": tool_outputs}


def _dedupe_workers(items: list[str]) -> list[str]:
    allowed = list(WORKERS)
    return [name for name in allowed if name in items]


@task(retry_policy=TRANSIENT_RETRY)
def route_request(request: str, available_inputs: dict, run_all_specialists: bool = False) -> dict:
    plan = router_llm.invoke(
        "You are the Talaqah supervisor. Select one or more specialist workers. "
        "Select content_slide_agent for a presentation/deck; delivery_agent for transcript/audio/video; "
        "confidence_agent for confidence, shyness, anxiety, rehearsal planning, or when a complete plan is requested. "
        "Return all needed workers, not only one.\n\n"
        f"Available inputs: {json.dumps(available_inputs, ensure_ascii=False)}\n"
        f"Run all specialists requested by UI: {bool(run_all_specialists)}\n"
        f"User request: {request}"
    ).model_dump()

    selected = _dedupe_workers(plan.get("destinations", []))
    safeguards = []
    if run_all_specialists:
        selected = list(WORKERS)
        safeguards.append("user_selected_full_team")
    else:
        if available_inputs.get("presentation") and "content_slide_agent" not in selected:
            selected.append("content_slide_agent")
            safeguards.append("presentation_input_requires_content_worker")
        if any(available_inputs.get(key) for key in ("transcript", "audio", "video")) and "delivery_agent" not in selected:
            selected.append("delivery_agent")
            safeguards.append("rehearsal_input_requires_delivery_worker")
    selected = _dedupe_workers(selected) or ["confidence_agent"]
    plan["destinations"] = selected
    plan["execution_safeguards"] = safeguards
    return plan


def _enforce_worker_tool_contract(compact: dict, destination: str, request: str, file_context: dict) -> dict:
    """Guarantee that supplied files cannot be silently ignored if a model omits a required tool call."""
    called = {item.get("name") for item in compact.get("tool_calls", [])}
    required = []
    if destination == "content_slide_agent":
        required.append((retrieve_guidelines, {"query": request + " slide design and presentation structure", "k": 4}))
        if file_context.get("presentation_path"):
            required.append((analyze_slide_deck, {"presentation_path": file_context["presentation_path"]}))
    elif destination == "delivery_agent":
        required.append((retrieve_guidelines, {"query": request + " speaking pace pauses fillers and vocal delivery", "k": 4}))
        common = {
            "transcript_path": file_context.get("transcript_path") or "",
            "actual_duration_minutes": float(file_context.get("actual_duration_minutes") or 0),
        }
        if file_context.get("audio_path"):
            required.append((analyze_audio_file, {"audio_path": file_context["audio_path"], **common}))
        if file_context.get("video_path"):
            required.append((analyze_video_file, {"video_path": file_context["video_path"], **common}))
        if not file_context.get("audio_path") and not file_context.get("video_path") and file_context.get("transcript_path"):
            required.append((analyze_transcript_file, {
                "transcript_path": file_context["transcript_path"],
                "actual_duration_minutes": float(file_context.get("actual_duration_minutes") or 0),
            }))
    elif destination == "confidence_agent":
        required.append((retrieve_guidelines, {"query": request + " confidence anxiety rehearsal gradual exposure", "k": 4}))

    enforced = []
    for tool_object, arguments in required:
        if tool_object.name in called:
            continue
        output = tool_object.invoke(arguments)
        compact["tool_calls"].append({
            "name": tool_object.name,
            "args": arguments,
            "enforced_worker_contract": True,
        })
        compact["tool_outputs"].append({
            "tool": tool_object.name,
            "output": str(output)[:10000],
        })
        enforced.append(tool_object.name)
    compact["enforced_tool_contract"] = enforced
    return compact


@task(retry_policy=TRANSIENT_RETRY)
def invoke_worker(destination: str, worker_prompt: str, request: str, file_context: dict) -> dict:
    result = WORKERS[destination].invoke({"messages": [{"role": "user", "content": worker_prompt}]})
    compact = compact_agent_result(destination, result)
    return _enforce_worker_tool_contract(compact, destination, request, file_context)


@task(retry_policy=TRANSIENT_RETRY)
def structure_coaching_report(request: str, plan: dict, worker_results: dict, memory_context: dict) -> dict:
    prompt = (
        "Create one validated Talaqah coaching report by combining all specialist evidence. "
        "Do not invent observations. Keep deterministic measurements separate from qualitative audio/video observations. "
        "Copy numeric metrics exactly, cite retrieved filenames, mention which specialists contributed, and give practical actions.\n\n"
        f"User request: {request}\nSupervisor plan: {json.dumps(plan, ensure_ascii=False)}\n"
        f"Learner context: {json.dumps(memory_context, ensure_ascii=False)}\n"
        f"All worker results: {json.dumps(worker_results, ensure_ascii=False)}"
    )
    return report_llm.invoke(prompt).model_dump()


@task(retry_policy=TRANSIENT_RETRY)
def evaluate_coaching_report(report: dict, request: str) -> dict:
    prompt = (
        "Evaluate this Talaqah report strictly. It must answer the request, combine all selected specialists, "
        "use tool evidence, avoid unsupported audio/visual claims, cite source filenames, and be actionable.\n\n"
        f"Request: {request}\nReport: {json.dumps(report, ensure_ascii=False)}"
    )
    return evaluator_llm.invoke(prompt).model_dump()


@task(retry_policy=TRANSIENT_RETRY)
def optimize_coaching_report(report: dict, evaluation: dict, request: str) -> dict:
    prompt = (
        "Revise the Talaqah report using every evaluator issue. Preserve exact measured metrics and sources. "
        "Do not invent evidence.\n\n"
        f"Request: {request}\nDraft: {json.dumps(report, ensure_ascii=False)}\n"
        f"Evaluator feedback: {json.dumps(evaluation, ensure_ascii=False)}"
    )
    return optimizer_llm.invoke(prompt).model_dump()


@task
def publish_approved_report(report: dict, user_id: str) -> dict:
    safe_user = re.sub(r"[^A-Za-z0-9_.-]+", "_", user_id)
    json_path = OUTPUT_DIR / f"{safe_user}_approved_report.json"
    md_path = OUTPUT_DIR / f"{safe_user}_approved_report.md"
    json_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    markdown_lines = [
        "# Talaqah AI — Approved Coaching Report", "", report.get("overall_assessment", ""), "",
        "## Strengths", *[f"- {x}" for x in report.get("strengths", [])], "",
        "## Areas for Improvement", *[f"- {x}" for x in report.get("areas_for_improvement", [])], "",
        "## Practice Actions", *[f"- {x}" for x in report.get("practice_actions", [])], "",
        "## Sources", *[f"- {x}" for x in report.get("sources", [])],
    ]
    md_path.write_text("\n".join(markdown_lines), encoding="utf-8")
    return {"json_path": str(json_path), "markdown_path": str(md_path)}


print("✅ Multi-worker Functional API tasks and quota-aware RetryPolicy registered.")


✅ Multi-worker Functional API tasks and quota-aware RetryPolicy registered.


## 9. Main capstone workflow

The report is not published until a human approves or edits it. All random/API work is inside `@task`, so completed work is checkpointed and not repeated unnecessarily when the workflow resumes.


In [14]:
checkpointer = InMemorySaver()
long_term_store = InMemoryStore()


@entrypoint(checkpointer=checkpointer, store=long_term_store)
def speakup_workflow(
    inputs: dict,
    *,
    previous: Any = None,
    store: BaseStore,
    config: RunnableConfig,
) -> entrypoint.final[dict, dict]:
    payload = dict(inputs)
    user_id = str(payload.get("user_id", "anonymous"))
    request = str(payload.get("request", "")).strip()
    if not request:
        raise ValueError("request cannot be empty")

    previous_state = previous if isinstance(previous, dict) else {}
    turn_count = int(previous_state.get("turn_count", 0)) + 1
    namespace = ("talaqah_users", user_id)
    memory_update = payload.get("remember")
    if isinstance(memory_update, dict) and memory_update:
        store.put(namespace, "learner_profile", memory_update)
    stored_items = store.search(namespace)
    memory_context = stored_items[-1].value if stored_items else {}

    if payload.get("mode") == "memory_only":
        state = {
            "status": "memory_checked", "user_id": user_id,
            "thread_id": config["configurable"]["thread_id"],
            "turn_count": turn_count, "long_term_memory": memory_context,
        }
        return entrypoint.final(value=state, save=state)

    transcript_value = payload.get("transcript_path")
    audio_value = payload.get("audio_path")
    video_value = payload.get("video_path")
    presentation_value = payload.get("presentation_path")
    duration_value = payload.get("actual_duration_minutes")
    if duration_value in (None, "", 0, 0.0):
        duration_value = _probe_media_duration_minutes(audio_value) or _probe_media_duration_minutes(video_value)

    available_inputs = {
        "transcript": bool(transcript_value),
        "audio": bool(audio_value),
        "video": bool(video_value),
        "presentation": bool(presentation_value),
        "duration": bool(duration_value),
    }
    plan = route_request(
        request,
        available_inputs,
        bool(payload.get("run_all_specialists", False)),
    ).result()

    if "delivery_agent" in plan["destinations"]:
        has_media = bool(audio_value or video_value)
        has_transcript_metrics = bool(transcript_value and duration_value)
        if not has_media and not has_transcript_metrics:
            correction = interrupt({
                "type": "user_fixable_input_error",
                "action": "Upload audio/video, or provide a transcript together with its measured duration.",
                "missing": ["audio_or_video OR transcript_and_duration"],
                "route": plan,
            })
            if not isinstance(correction, dict):
                raise ValueError("Resume input must be a dictionary with corrected values.")
            transcript_value = correction.get("transcript_path", transcript_value)
            audio_value = correction.get("audio_path", audio_value)
            video_value = correction.get("video_path", video_value)
            duration_value = correction.get("actual_duration_minutes", duration_value)

    common_prompt = (
        f"User request: {request}\n"
        f"Transcript path: {transcript_value or 'not supplied'}\n"
        f"Audio path: {audio_value or 'not supplied'}\n"
        f"Video path: {video_value or 'not supplied'}\n"
        f"Actual duration in minutes (media tools should prefer measured media duration): {duration_value or 'not supplied'}\n"
        f"Presentation path: {presentation_value or 'not supplied'}\n"
        f"Learner context: {json.dumps(memory_context, ensure_ascii=False)}"
    )

    # Functional API fan-out/fan-in: all selected specialists are scheduled before any result is awaited.
    file_context = {
        "transcript_path": transcript_value,
        "audio_path": audio_value,
        "video_path": video_value,
        "presentation_path": presentation_value,
        "actual_duration_minutes": duration_value,
    }
    worker_futures = {
        destination: invoke_worker(destination, common_prompt, request, file_context)
        for destination in plan["destinations"]
    }
    worker_results = {
        destination: future.result()
        for destination, future in worker_futures.items()
    }

    draft_report = structure_coaching_report(request, plan, worker_results, memory_context).result()
    initial_evaluation = evaluate_coaching_report(draft_report, request).result()
    optimized_report = optimize_coaching_report(draft_report, initial_evaluation, request).result()
    final_evaluation = evaluate_coaching_report(optimized_report, request).result()

    final_report = optimized_report
    approval_status = "not_required"
    if payload.get("requires_approval", True):
        human_decision = interrupt({
            "type": "human_review_before_publication",
            "action": "Approve, edit, or reject the final coaching report.",
            "allowed_actions": ["approve", "edit", "reject"],
            "draft_report": final_report,
            "final_evaluation": final_evaluation,
            "route": plan,
            "workers": worker_results,
        })
        if not isinstance(human_decision, dict):
            raise ValueError("Human decision must be a dictionary.")
        action = human_decision.get("action")
        if action == "reject":
            state = {
                "status": "rejected_by_human", "turn_count": turn_count,
                "route": plan, "workers": worker_results, "report": final_report,
            }
            return entrypoint.final(value=state, save=state)
        if action == "edit":
            final_report = CoachingReport.model_validate(human_decision.get("edited_report")).model_dump()
            approval_status = "edited_and_approved"
        elif action == "approve":
            approval_status = "approved"
        else:
            raise ValueError("Human action must be approve, edit, or reject.")

    publication = publish_approved_report(final_report, user_id).result()
    state = {
        "status": "completed", "approval_status": approval_status,
        "user_id": user_id, "thread_id": config["configurable"]["thread_id"],
        "turn_count": turn_count, "route": plan, "workers": worker_results,
        "worker": next(iter(worker_results.values())),
        "initial_evaluation": initial_evaluation, "final_evaluation": final_evaluation,
        "report": final_report, "publication": publication,
        "long_term_memory": memory_context,
    }
    return entrypoint.final(value=state, save=state)


print("✅ Multi-agent Talaqah workflow compiled with parallel specialists, memory, evaluation, and HITL.")


✅ Multi-agent Talaqah workflow compiled with parallel specialists, memory, evaluation, and HITL.


## 10. Evidence A — LLM routing to one or more workers

The destinations, reason, and confidence below come from the structured `RoutePlan`. There is no keyword router. Input-aware safeguards only ensure that a supplied presentation or rehearsal file cannot be silently ignored.


In [15]:
@entrypoint()
def routing_evidence_workflow(cases: list[dict]) -> list[dict]:
    futures = [route_request(case["request"], case["inputs"], False) for case in cases]
    return [future.result() for future in futures]


routing_cases = [
    {"request": "Analyze my slide deck and reduce crowded slides.", "inputs": {"presentation": True}},
    {"request": "Measure filler words and delivery from my audio rehearsal.", "inputs": {"audio": True}},
    {"request": "I feel shy in front of an audience; build a gradual confidence plan.", "inputs": {}},
]
routing_results = routing_evidence_workflow.invoke(routing_cases)

for case, result in zip(routing_cases, routing_results):
    print("REQUEST:", case["request"])
    print("DESTINATIONS:", result["destinations"])
    print("REASON:", result["reason"])
    print("CONFIDENCE:", result["confidence"])
    print("-" * 70)

assert len({destination for item in routing_results for destination in item["destinations"]}) >= 2
print("✅ LLM multi-destination routing evidence captured.")


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


REQUEST: Analyze my slide deck and reduce crowded slides.
DESTINATIONS: ['content_slide_agent']
REASON: The user wants an analysis of their slide deck and help to reduce crowded slides, which falls directly under the content_slide_agent's expertise.
CONFIDENCE: 1.0
----------------------------------------------------------------------
REQUEST: Measure filler words and delivery from my audio rehearsal.
DESTINATIONS: ['delivery_agent', 'confidence_agent']
REASON: The user wants to measure filler words and delivery from an audio rehearsal, which requires delivery analysis and confidence assessment.
CONFIDENCE: 0.95
----------------------------------------------------------------------
REQUEST: I feel shy in front of an audience; build a gradual confidence plan.
DESTINATIONS: ['confidence_agent']
REASON: The user feels shy in front of an audience and has requested a gradual confidence plan, which falls under the purview of the confidence agent.
CONFIDENCE: 1.0
-----------------------------

## 11. Evidence B — short-term state and cross-thread long-term memory

- The same `thread_id` must increment `turn_count` from 1 to 2.
- A different thread for the same `user_id` must still retrieve the same learner profile from the separate Store.


In [16]:
memory_profile = {
    "preferred_language": "English",
    "main_goal": "reduce presentation anxiety",
    "known_challenge": "filler words during transitions",
}

same_thread_config = {"configurable": {"thread_id": "memory-thread-A"}}
memory_turn_1 = speakup_workflow.invoke({
    "mode": "memory_only",
    "user_id": "layan",
    "request": "Remember my presentation preferences.",
    "remember": memory_profile,
}, config=same_thread_config)

memory_turn_2 = speakup_workflow.invoke({
    "mode": "memory_only",
    "user_id": "layan",
    "request": "Read my saved presentation preferences.",
}, config=same_thread_config)

different_thread_config = {"configurable": {"thread_id": "memory-thread-B"}}
cross_thread_result = speakup_workflow.invoke({
    "mode": "memory_only",
    "user_id": "layan",
    "request": "Read my saved presentation preferences in a new thread.",
}, config=different_thread_config)

print("Same thread turn counts:", memory_turn_1["turn_count"], "→", memory_turn_2["turn_count"])
print("Different thread ID:", cross_thread_result["thread_id"])
print("Cross-thread memory:", json.dumps(cross_thread_result["long_term_memory"], indent=2))

assert memory_turn_1["turn_count"] == 1
assert memory_turn_2["turn_count"] == 2
assert cross_thread_result["long_term_memory"] == memory_profile
print("✅ Short-term and cross-thread long-term memory tests passed.")

Same thread turn counts: 1 → 2
Different thread ID: memory-thread-B
Cross-thread memory: {
  "preferred_language": "English",
  "main_goal": "reduce presentation anxiety",
  "known_challenge": "filler words during transitions"
}
✅ Short-term and cross-thread long-term memory tests passed.


## 12. Evidence C — user-fixable error with interrupt and resume

This lightweight test demonstrates the error strategy without hiding the error or relying on a comment. It pauses, receives corrected user input, and completes.


In [17]:
@entrypoint(checkpointer=checkpointer)
def input_repair_demo(inputs: dict) -> dict:
    duration = inputs.get("actual_duration_minutes")
    if not duration:
        correction = interrupt({
            "type": "user_fixable_input_error",
            "missing": "actual_duration_minutes",
            "action": "Provide the measured recording duration.",
        })
        duration = correction["actual_duration_minutes"]
    if float(duration) <= 0:
        raise ValueError("Duration must be greater than zero.")
    return {"status": "repaired", "actual_duration_minutes": float(duration)}


repair_config = {"configurable": {"thread_id": "repair-demo-1"}}
repair_paused = input_repair_demo.invoke({}, config=repair_config)
print("PAUSED:", repair_paused["__interrupt__"][0].value)

repair_completed = input_repair_demo.invoke(
    Command(resume={"actual_duration_minutes": 3.0}),
    config=repair_config,
)
print("RESUMED:", repair_completed)
assert repair_completed["status"] == "repaired"
print("✅ User-fixable error was paused and resumed successfully.")

PAUSED: {'type': 'user_fixable_input_error', 'missing': 'actual_duration_minutes', 'action': 'Provide the measured recording duration.'}
RESUMED: {'status': 'repaired', 'actual_duration_minutes': 3.0}
✅ User-fixable error was paused and resumed successfully.


## 13. Evidence D — full agent run, real tool calls, Evaluator–Optimizer, and human approval

`ACTUAL_DURATION_MINUTES` must be the stopwatch duration of the recording represented by the transcript. The demo uses 3.0 minutes; replace it if your recorded rehearsal has a different measured duration.


In [18]:
ACTUAL_DURATION_MINUTES = 3.0

full_run_config = {"configurable": {"thread_id": "talaqah-full-demo-1"}}
full_run_input = {
    "user_id": "layan",
    "request": (
        "Analyze my slides and rehearsal delivery, measure pace and filler words, "
        "and give me a complete confidence practice plan."
    ),
    "transcript_path": str(transcript_path),
    "audio_path": str(audio_path) if audio_path else None,
    "video_path": str(video_path) if video_path else None,
    "actual_duration_minutes": ACTUAL_DURATION_MINUTES,
    "presentation_path": str(presentation_path),
    "run_all_specialists": True,
    "requires_approval": True,
    "remember": memory_profile,
}

full_run_paused = speakup_workflow.invoke(full_run_input, config=full_run_config)
approval_payload = full_run_paused["__interrupt__"][0].value

print("PAUSED BEFORE PUBLICATION:", approval_payload["type"])
print("SELECTED SPECIALISTS:", approval_payload["route"]["destinations"])
print("EVALUATOR SCORE BEFORE APPROVAL:", approval_payload["final_evaluation"]["score"])
print("DRAFT ASSESSMENT:", approval_payload["draft_report"]["overall_assessment"])
print("✅ interrupt() captured; nothing has been published yet.")


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

PAUSED BEFORE PUBLICATION: human_review_before_publication
SELECTED SPECIALISTS: ['content_slide_agent', 'delivery_agent', 'confidence_agent']
EVALUATOR SCORE BEFORE APPROVAL: 5
DRAFT ASSESSMENT: The presentation deck (`talaqah_demo_presentation.pptx`) features 4 well-balanced slides with clean word counts ranging from 9 to 12 words. The rehearsal delivery (`rehearsal_transcript.txt`) lasted 3.0 minutes with a speaking rate of 123 WPM (controlled pace) and a total of 5 filler words (`um`: 3, `basically`: 1, `you know`: 1) primarily clustering around slide transitions. The content slide agent, delivery agent, and confidence agent all contributed to this comprehensive coaching report.
✅ interrupt() captured; nothing has been published yet.


In [19]:
full_run_result = speakup_workflow.invoke(
    Command(resume={"action": "approve"}),
    config=full_run_config,
)

print("FINAL STATUS:", full_run_result["status"])
print("APPROVAL:", full_run_result["approval_status"])
print("ROUTE PLAN:", full_run_result["route"])
print("\nREAL MODEL-CHOSEN TOOL CALLS BY SPECIALIST:")
for worker_name, worker_result in full_run_result["workers"].items():
    print(f"\n[{worker_name}]")
    for tool_call in worker_result["tool_calls"]:
        print("-", tool_call["name"], tool_call["args"])

print("\nEVALUATOR–OPTIMIZER EVIDENCE:")
print("Initial evaluation:", full_run_result["initial_evaluation"])
print("Final evaluation:", full_run_result["final_evaluation"])
print("\nPUBLISHED FILES:", full_run_result["publication"])

assert full_run_result["status"] == "completed"
assert full_run_result["approval_status"] == "approved"
assert set(full_run_result["workers"]) == set(WORKERS), "Full-team demo did not execute all specialists."
assert all(result["tool_calls"] for result in full_run_result["workers"].values()), "A specialist made no tool calls."
print("✅ All three specialists executed; Command(resume=...) approved and published the report.")


FINAL STATUS: completed
APPROVAL: approved
ROUTE PLAN: {'destinations': ['content_slide_agent', 'delivery_agent', 'confidence_agent'], 'reason': 'The user requested an analysis of slides, rehearsal delivery, metrics like pace and filler words, and a complete confidence practice plan, which requires all three specialist agents.', 'confidence': 1.0, 'execution_safeguards': ['user_selected_full_team']}

REAL MODEL-CHOSEN TOOL CALLS BY SPECIALIST:

[content_slide_agent]
- retrieve_guidelines {'query': 'filler words presentation anxiety slide design pace rehearsal'}
- analyze_slide_deck {'presentation_path': '/content/talaqah_inputs/talaqah_demo_presentation.pptx'}

[delivery_agent]
- retrieve_guidelines {'query': 'reduce presentation anxiety filler words during transitions pacing confidence practice plan'}
- analyze_transcript_file {'transcript_path': '/content/talaqah_inputs/rehearsal_transcript.txt', 'actual_duration_minutes': 3}

[confidence_agent]
- retrieve_guidelines {'k': 3, 'query'

## 14. Evidence E — LangSmith trace inspection

This cell does not merely state that tracing is enabled. It reads the actual traces and prints an observed bottleneck or error from this run.


In [20]:
wait_for_all_tracers()
trace_runs = list(langsmith_client.list_runs(project_name=LANGSMITH_PROJECT, limit=100))

if not trace_runs:
    raise RuntimeError("No LangSmith traces were found. Check the key and tracing environment variables.")

completed_runs = [run for run in trace_runs if run.start_time and run.end_time]
if not completed_runs:
    raise RuntimeError("Traces exist but no completed run was available for latency analysis.")

def run_duration_seconds(run) -> float:
    return (run.end_time - run.start_time).total_seconds()

slowest_run = max(completed_runs, key=run_duration_seconds)
error_runs = [run for run in trace_runs if getattr(run, "error", None)]

trace_insight = {
    "project": LANGSMITH_PROJECT,
    "runs_inspected": len(trace_runs),
    "slowest_step": slowest_run.name,
    "slowest_step_seconds": round(run_duration_seconds(slowest_run), 3),
    "error_count": len(error_runs),
    "observed_insight": (
        f"The trace showed that '{slowest_run.name}' was the slowest inspected step "
        f"at {run_duration_seconds(slowest_run):.3f} seconds. "
        f"It also showed {len(error_runs)} traced error(s)."
    ),
}

print(json.dumps(trace_insight, indent=2, ensure_ascii=False))
print("✅ LangSmith tracing is genuinely active and an actual trace insight was recorded.")

{
  "project": "Talaqah-AI-Capstone-August-2026",
  "runs_inspected": 100,
  "slowest_step": "invoke_worker",
  "slowest_step_seconds": 98.526,
  "error_count": 1,
  "observed_insight": "The trace showed that 'invoke_worker' was the slowest inspected step at 98.526 seconds. It also showed 1 traced error(s)."
}
✅ LangSmith tracing is genuinely active and an actual trace insight was recorded.


## 15. Automated submission evidence summary

All assertions below must print `PASS`. If any assertion fails, do not submit until it is fixed and the notebook is rerun from the beginning.


In [21]:
all_tool_calls = [
    call
    for worker_result in full_run_result["workers"].values()
    for call in worker_result["tool_calls"]
]
capstone_evidence = {
    "1_agent_fundamentals_and_real_tools": bool(all_tool_calls),
    "1_audio_video_tools_available": {analyze_audio_file.name, analyze_video_file.name}.issubset(
        {tool.name for tool in DELIVERY_TOOLS}
    ),
    "2_multi_agent_supervisor_and_parallel_workers": set(full_run_result["workers"]) == set(WORKERS),
    "3_rag_pipeline": len(documents) >= 2 and len(chunks) > len(documents) and bool(rag_test["results"]),
    "4_short_term_state": memory_turn_2["turn_count"] == 2,
    "4_long_term_cross_thread": cross_thread_result["long_term_memory"] == memory_profile,
    "5_human_in_the_loop": full_run_result["approval_status"] == "approved",
    "6_functional_api_and_errors": repair_completed["status"] == "repaired",
    "7_evaluator_optimizer": bool(full_run_result["initial_evaluation"]) and bool(full_run_result["final_evaluation"]),
    "8_langsmith": len(trace_runs) > 0 and bool(trace_insight["observed_insight"]),
}

for requirement, passed in capstone_evidence.items():
    print(f"{'PASS' if passed else 'FAIL'} — {requirement}")

assert all(capstone_evidence.values()), "At least one capstone requirement failed."
print("\n✅ Every capstone section has visible execution evidence.")


PASS — 1_agent_fundamentals_and_real_tools
PASS — 1_audio_video_tools_available
PASS — 2_multi_agent_supervisor_and_parallel_workers
PASS — 3_rag_pipeline
PASS — 4_short_term_state
PASS — 4_long_term_cross_thread
PASS — 5_human_in_the_loop
PASS — 6_functional_api_and_errors
PASS — 7_evaluator_optimizer
PASS — 8_langsmith

✅ Every capstone section has visible execution evidence.


## 16. Talaqah live website — agents + microphone + camera

This cell launches the real Talaqah/Talaqah workflow as a Gradio website. It adds live agent-status cards, microphone rehearsal, webcam/video rehearsal, the real Evaluator–Optimizer flow, and HITL approval. Audio/video preprocessing uses the same `GEMINI_API_KEY`; LangSmith continues to use the notebook's existing tracing setup.


In [22]:
# ============================================================
# TALAAQAH LIVE UI — TRANSCRIPT + SLIDES + AUDIO + VIDEO
# ============================================================
import gradio as gr
from copy import deepcopy


def _get_path(value):
    if value is None:
        return None
    return value if isinstance(value, str) else getattr(value, "name", None)


def _escape_html(value):
    return (str(value or "").replace("&", "&amp;").replace("<", "&lt;")
            .replace(">", "&gt;").replace('"', "&quot;").replace("'", "&#039;"))


def _list_html(items):
    return "".join(f"<div class='report-item'>✓ {_escape_html(x)}</div>" for x in (items or [])) or "<p class='muted'>No items available.</p>"


def _metrics_html(metrics):
    cards = []
    for metric in metrics or []:
        value = metric if isinstance(metric, dict) else metric.model_dump()
        cards.append(f"<div class='metric-card'><div class='metric-name'>{_escape_html(value.get('name'))}</div><div class='metric-value'>{_escape_html(value.get('value'))}</div></div>")
    return f"<div class='metrics-grid'>{''.join(cards)}</div>" if cards else ""


def render_report(result):
    if result is None:
        return "<div class='empty-report'><div class='empty-icon'>🎯</div><h2>Talaqah Coaching Report</h2><p>Your report will appear here after analysis.</p></div>"
    report = result.get("report") or result.get("final_report") or result.get("draft_report") or result if isinstance(result, dict) else result
    if hasattr(report, "model_dump"):
        report = report.model_dump()
    if not isinstance(report, dict):
        report = {"overall_assessment": str(report)}
    return f"""
    <div class='report-wrapper'>
      <div class='report-header'><div><div class='report-label'>🎯 TALAAQAH COACHING REPORT</div><h2>Your AI Coaching Report</h2></div><div class='completed-badge'>✓ Ready</div></div>
      <div class='assessment-card'><div class='section-title'>✨ Overall Assessment</div><p>{_escape_html(report.get('overall_assessment'))}</p></div>
      {_metrics_html(report.get('metrics'))}
      <div class='report-section'><div class='section-title'>💪 Strengths</div><div class='report-list'>{_list_html(report.get('strengths'))}</div></div>
      <div class='report-section'><div class='section-title'>🎯 Areas for Improvement</div><div class='report-list'>{_list_html(report.get('areas_for_improvement'))}</div></div>
      <div class='report-section'><div class='section-title'>🚀 Practice Actions</div><div class='report-list'>{_list_html(report.get('practice_actions'))}</div></div>
      <div class='report-section'><div class='section-title'>📚 Sources</div><div class='report-list'>{_list_html(report.get('sources'))}</div></div>
    </div>"""


AGENT_META = {
    "router": {"code": "AG-01", "title": "الموجّه", "icon": "🧭", "role": "يفهم الطلب ويختار جميع الوكلاء المناسبين"},
    "content_slide": {"code": "AG-02", "title": "المحتوى والشرائح", "icon": "🖥️", "role": "يحلل العرض والشرائح باستخدام الأدوات"},
    "delivery": {"code": "AG-03", "title": "مدرب الإلقاء", "icon": "🎤", "role": "يحلل النص والصوت والفيديو وقياسات الإلقاء"},
    "confidence": {"code": "AG-04", "title": "مدرب الثقة", "icon": "🌱", "role": "يبني خطة تدريب تدريجية مدعومة بالمصادر"},
    "report": {"code": "AG-05", "title": "منظّم التقرير", "icon": "📝", "role": "يدمج نتائج جميع الوكلاء في تقرير واحد"},
    "evaluator": {"code": "AG-06", "title": "المقيّم", "icon": "📊", "role": "يفحص جودة التقرير وارتباطه بالأدلة"},
    "optimizer": {"code": "AG-07", "title": "المُحسّن", "icon": "🛠️", "role": "يحسن التقرير بناءً على التقييم"},
    "human": {"code": "HITL", "title": "الموافقة البشرية", "icon": "✅", "role": "يراجع المستخدم النسخة النهائية قبل النشر"},
}
WORKER_STAGE = {"content_slide_agent": "content_slide", "delivery_agent": "delivery", "confidence_agent": "confidence"}


def new_board_state():
    return {key: {"status": "waiting", "detail": "بانتظار المرحلة السابقة"} for key in AGENT_META}


def _set_stage(board, stage, status, detail):
    if stage in board:
        board[stage] = {"status": status, "detail": detail}


def _sync_selected_workers(board, destinations, status="running"):
    selected = set(destinations or [])
    for worker, stage in WORKER_STAGE.items():
        if worker in selected:
            _set_stage(board, stage, status, "جاري استخدام الأدوات وتحليل المدخلات…" if status == "running" else "اكتمل التحليل بنجاح")
        else:
            _set_stage(board, stage, "skipped", "غير مطلوب لهذا الطلب")


def _task_info(data):
    if not isinstance(data, dict):
        return "", None, None, False
    name = str(data.get("name") or data.get("task_name") or data.get("node") or "")
    result = data.get("result", data.get("output"))
    error = data.get("error")
    finished = error is not None or "result" in data or "output" in data or str(data.get("status", "")).lower() in {"done", "completed", "success", "failed", "error"}
    return name.lower(), result, error, finished


def _update_board_from_task(board, data):
    name, result, error, finished = _task_info(data)
    if "route_request" in name:
        if error:
            _set_stage(board, "router", "error", str(error))
        elif finished and isinstance(result, dict):
            destinations = result.get("destinations", [])
            # Some LangGraph task-stream events contain a partial/empty result.
            # Never overwrite the UI's requested full-team state from such an event.
            if destinations:
                _set_stage(board, "router", "done", "تم اختيار: " + ", ".join(destinations))
                _sync_selected_workers(board, destinations, "running")
        else:
            _set_stage(board, "router", "running", "جاري فهم الطلب وبناء خطة التنفيذ…")
        return
    if "invoke_worker" in name and finished and isinstance(result, dict):
        stage = WORKER_STAGE.get(result.get("worker"))
        if stage:
            tools = [x.get("name") for x in result.get("tool_calls", []) if isinstance(x, dict)]
            _set_stage(board, stage, "error" if error else "done", str(error) if error else "اكتمل • " + ", ".join(filter(None, tools[:4])))
        return
    task_stage = {
        "structure_coaching_report": "report",
        "evaluate_coaching_report": "evaluator",
        "optimize_coaching_report": "optimizer",
        "publish_approved_report": "human",
    }
    for task_name, stage in task_stage.items():
        if task_name in name:
            if stage == "report" and not finished:
                for worker_stage in WORKER_STAGE.values():
                    if board[worker_stage]["status"] == "running":
                        _set_stage(board, worker_stage, "done", "اكتمل التحليل بنجاح")
            if error:
                _set_stage(board, stage, "error", str(error))
            elif finished:
                detail = "اكتملت المرحلة بنجاح"
                if stage == "evaluator" and isinstance(result, dict) and result.get("score"):
                    detail = f"اكتمل التقييم • {result['score']}/5"
                _set_stage(board, stage, "done", detail)
            else:
                _set_stage(board, stage, "running", "يعمل الآن على البيانات الفعلية…")
            return


def _finalize_board(board, route, human_waiting=True):
    destinations = (route or {}).get("destinations", [])
    _set_stage(board, "router", "done", "تم اختيار: " + ", ".join(destinations))
    _sync_selected_workers(board, destinations, "done")
    for stage in ("report", "evaluator", "optimizer"):
        _set_stage(board, stage, "done", "اكتملت المرحلة بنجاح")
    _set_stage(board, "human", "waiting" if human_waiting else "done", "بانتظار موافقتك قبل النشر" if human_waiting else "تم الاعتماد والنشر")


def _person_markup(index, icon):
    hair = ["hair-a", "hair-b", "hair-c", "hair-d"][index % 4]
    shirt = ["shirt-cyan", "shirt-purple", "shirt-teal", "shirt-blue"][index % 4]
    prop = ["⌨️", "📑", "🎙️", "💬", "📝", "📈", "⚙️", "👍"][index % 8]
    return f"<div class='person-scene'><div class='status-glow'></div><div class='person-head {hair}'><span class='agent-icon'>{icon}</span></div><div class='person-body {shirt}'></div><div class='person-arm arm-left'></div><div class='person-arm arm-right'></div><div class='desk-top'></div><div class='laptop'><span>{prop}</span></div></div>"


def render_agent_board(board=None):
    board = deepcopy(board or new_board_state())
    cards = []
    status_ar = {"waiting": "قيد الانتظار", "running": "يعمل الآن", "done": "مكتمل", "skipped": "غير مستخدم", "error": "خطأ"}
    for i, (key, meta) in enumerate(AGENT_META.items()):
        data = board[key]
        cards.append(f"<div class='agent-card status-{data['status']}'><div class='agent-bubble'>{_escape_html(data['detail'])}</div>{_person_markup(i, meta['icon'])}<div class='agent-name'>{meta['title']}</div><div class='agent-role'>{meta['role']}</div><div class='agent-footer'><span class='agent-code'>{meta['code']}</span><span class='agent-status'><i></i>{status_ar[data['status']]}</span></div></div>")
    return f"<section class='team-board' dir='rtl'><div class='team-board-head'><div><span class='team-kicker'>LIVE AGENT WORKFLOW</span><h2>🧠 فريق طلاقة أثناء العمل</h2><p>الحالات مأخوذة من مهام LangGraph الفعلية.</p></div></div><div class='agents-grid'>{''.join(cards)}</div></section>"


_PENDING_UI_RUNS = {}


def _extract_interrupt(value):
    if not isinstance(value, dict):
        return None
    values = value.get("__interrupt__") or value.get("interrupt")
    if not values:
        return None
    return getattr(values[0], "value", values[0])


def run_talaqah_live(topic, request, transcript_file, audio_recording, video_recording, presentation_file, duration):
    board = new_board_state()
    request = (request or "").strip()
    if not request:
        yield render_agent_board(board), "<div class='error-card'>⚠️ اكتبي المطلوب تحليله أولًا.</div>", None, gr.update(visible=False), gr.update(visible=False)
        return
    transcript_path = _get_path(transcript_file)
    audio_path_value = _get_path(audio_recording)
    video_path_value = _get_path(video_recording)
    presentation_path_value = _get_path(presentation_file)
    duration_value = float(duration or 0)
    if duration_value <= 0:
        duration_value = _probe_media_duration_minutes(audio_path_value) or _probe_media_duration_minutes(video_path_value)
    enriched_request = f"Presentation topic: {(topic or 'not supplied').strip()}\nUser request: {request}"
    run_id = uuid.uuid4().hex
    config = {"configurable": {"thread_id": f"talaqah-ui-{run_id}"}}
    # The capstone demo always runs all three specialist agents. This avoids a
    # browser/component value mismatch and provides complete execution evidence.
    run_all_specialists = True
    payload = {
        "user_id": "talaqah_demo_user", "request": enriched_request,
        "transcript_path": transcript_path, "audio_path": audio_path_value,
        "video_path": video_path_value, "presentation_path": presentation_path_value,
        "actual_duration_minutes": duration_value,
        "run_all_specialists": True, "requires_approval": True,
    }
    _set_stage(board, "router", "running", "جاري فهم الطلب وبناء خطة التنفيذ…")
    _sync_selected_workers(board, list(WORKER_STAGE), "running")
    for stage in WORKER_STAGE.values():
        board[stage]["detail"] = "تم طلب الفريق الكامل • بانتظار اكتمال خطة الموجّه"
    yield render_agent_board(board), render_report(None), run_id, gr.update(visible=False), gr.update(visible=False)
    last_value = None
    try:
        try:
            stream = speakup_workflow.stream(payload, config=config, stream_mode=["tasks", "values"], version="v2")
            v2 = True
        except TypeError:
            stream = speakup_workflow.stream(payload, config=config, stream_mode=["tasks", "values"])
            v2 = False
        for chunk in stream:
            if v2 and isinstance(chunk, dict):
                mode, data = chunk.get("type"), chunk.get("data")
            elif isinstance(chunk, tuple) and len(chunk) == 2:
                mode, data = chunk
            else:
                mode, data = None, chunk
            if mode == "tasks":
                _update_board_from_task(board, data)
                yield render_agent_board(board), render_report(None), run_id, gr.update(visible=False), gr.update(visible=False)
            elif mode == "values":
                last_value = data
        result = last_value
        if result is None:
            try:
                result = getattr(speakup_workflow.get_state(config), "values", None)
            except Exception:
                result = None
        interrupt_value = _extract_interrupt(result)
        if interrupt_value is None:
            try:
                snapshot = speakup_workflow.get_state(config)
                for task_info in getattr(snapshot, "tasks", ()) or ():
                    interrupts = getattr(task_info, "interrupts", ()) or ()
                    if interrupts:
                        interrupt_value = getattr(interrupts[0], "value", interrupts[0])
                        break
            except Exception:
                pass
        if isinstance(interrupt_value, dict):
            if interrupt_value.get("type") == "user_fixable_input_error":
                _set_stage(board, "human", "waiting", "مدخلات ناقصة")
                missing = ", ".join(interrupt_value.get("missing", []))
                yield render_agent_board(board), f"<div class='error-card'>⚠️ Missing input: {_escape_html(missing)}</div>", None, gr.update(visible=False), gr.update(visible=False)
                return
            if interrupt_value.get("type") == "human_review_before_publication":
                _finalize_board(board, interrupt_value.get("route"), True)
                _PENDING_UI_RUNS[run_id] = {"config": config, "board": deepcopy(board), "draft": interrupt_value.get("draft_report")}
                yield render_agent_board(board), render_report(interrupt_value.get("draft_report")), run_id, gr.update(visible=True), gr.update(visible=True)
                return
        if isinstance(result, dict) and result.get("status") == "completed":
            _finalize_board(board, result.get("route"), False)
        yield render_agent_board(board), render_report(result), None, gr.update(visible=False), gr.update(visible=False)
    except Exception as exc:
        for stage, data in board.items():
            if data["status"] == "running":
                _set_stage(board, stage, "error", f"{type(exc).__name__}: {exc}")
        yield render_agent_board(board), f"<div class='error-card'>❌ {type(exc).__name__}: {_escape_html(exc)}<br><br>إذا كان الخطأ 429 انتظري دقيقة ثم أعيدي التشغيل؛ محدد السرعة وإعادة المحاولة مفعّلان.</div>", None, gr.update(visible=False), gr.update(visible=False)


def _resume_talaqah(run_id, action):
    pending = _PENDING_UI_RUNS.get(run_id or "")
    if not pending:
        return render_agent_board(new_board_state()), "<div class='error-card'>No pending review found.</div>", None, gr.update(visible=False), gr.update(visible=False)
    board = pending["board"]
    _set_stage(board, "human", "running", "جاري تنفيذ قرارك…")
    try:
        result = speakup_workflow.invoke(Command(resume={"action": action}), config=pending["config"])
        _set_stage(board, "human", "done" if action == "approve" else "skipped", "تم الاعتماد والنشر" if action == "approve" else "تم الرفض ولم يُنشر التقرير")
        _PENDING_UI_RUNS.pop(run_id, None)
        return render_agent_board(board), render_report(result), None, gr.update(visible=False), gr.update(visible=False)
    except Exception as exc:
        _set_stage(board, "human", "error", str(exc))
        return render_agent_board(board), f"<div class='error-card'>❌ {type(exc).__name__}: {_escape_html(exc)}</div>", run_id, gr.update(visible=True), gr.update(visible=True)


def approve_talaqah(run_id):
    return _resume_talaqah(run_id, "approve")


def reject_talaqah(run_id):
    return _resume_talaqah(run_id, "reject")


custom_css = r"""
body {
    background: radial-gradient(circle at 50% 0%, rgba(28,87,135,.25), transparent 38%), #030914 !important;
}
.gradio-container {
    max-width: 1450px !important;
    margin: auto !important;
    padding: 24px !important;
    background: transparent !important;
}

#talaqah-hero {
    background: linear-gradient(135deg, rgba(7,42,68,.96), rgba(10,16,48,.96));
    border: 1px solid rgba(75,211,255,.28);
    border-radius: 28px;
    padding: 42px 35px 38px;
    margin-bottom: 26px;
    text-align: center;
    box-shadow: 0 0 35px rgba(0,183,255,.07), inset 0 0 40px rgba(0,0,0,.12);
}
#talaqah-hero h1 {
    font-size: 42px !important;
    font-weight: 800 !important;
    margin-bottom: 12px !important;
    background: linear-gradient(90deg,#63e7ff,#fff,#a78bfa);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
#talaqah-hero .subtitle {font-size:21px;color:#a9bdd5;margin-bottom:8px}
#talaqah-hero .tagline {font-size:17px;color:#dce8f6}
.hero-line {height:4px;max-width:820px;margin:28px auto 0;border-radius:10px;background:linear-gradient(90deg,#1fcfff,#8b5cf6);box-shadow:0 0 15px rgba(31,207,255,.5)}

.talaqah-card, #talaqah-report {
    background: linear-gradient(145deg,rgba(8,20,42,.96),rgba(6,15,31,.96));
    border:1px solid rgba(83,188,255,.20);
    border-radius:24px;
    padding:28px;
    box-shadow:0 12px 45px rgba(0,0,0,.22);
}
.talaqah-card {min-height:520px}
#talaqah-report {min-height:520px}
.card-title {font-size:28px;font-weight:800;color:#f3f7fc;margin-bottom:16px}
.card-text {font-size:16px;line-height:1.7;color:#bdcce0;margin-bottom:24px}

#talaqah-input textarea {
    background:rgba(19,22,31,.95)!important;
    border:1px solid rgba(62,194,255,.34)!important;
    border-radius:15px!important;
    color:#f5f8ff!important;
    font-size:16px!important;
}
#talaqah-input textarea:focus {border-color:#43d8ff!important;box-shadow:0 0 0 1px #43d8ff,0 0 18px rgba(67,216,255,.15)!important}
#talaqah-input label {color:#e8eef8!important;font-weight:700!important}

#talaqah-run {
    margin-top:16px;min-height:54px!important;border-radius:15px!important;font-size:17px!important;font-weight:800!important;
    background:linear-gradient(90deg,#16c9ff,#705cff)!important;border:none!important;box-shadow:0 8px 25px rgba(40,166,255,.20)
}
#talaqah-run:hover {filter:brightness(1.08);transform:translateY(-1px);box-shadow:0 10px 30px rgba(40,166,255,.30)}

/* ---------- LIVE TEAM BOARD ---------- */
.team-board {
    margin: 28px 0;
    background: linear-gradient(145deg,rgba(6,17,37,.98),rgba(6,13,29,.98));
    border:1px solid rgba(83,188,255,.20);
    border-radius:26px;
    padding:26px;
    overflow:hidden;
    box-shadow:0 14px 50px rgba(0,0,0,.24);
}
.team-board-head {display:flex;justify-content:space-between;align-items:end;margin-bottom:22px}
.team-kicker {font-size:11px;letter-spacing:1.8px;color:#55dfff;font-weight:800}
.team-board h2 {margin:5px 0 5px;color:#f5f9ff;font-size:28px}
.team-board p {margin:0;color:#8fa7c1;font-size:14px}
.agents-grid {display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:18px}
.agent-card {
    position:relative;min-width:0;text-align:center;padding:70px 14px 15px;
    border:1px solid rgba(99,160,218,.15);border-radius:20px;
    background:linear-gradient(180deg,rgba(13,29,54,.92),rgba(8,18,35,.92));
    min-height:330px;transition:.2s ease;
}
.agent-bubble {
    position:absolute;top:12px;left:10px;right:10px;height:48px;overflow:auto;
    display:flex;align-items:center;justify-content:center;padding:7px 9px;box-sizing:border-box;
    border-radius:13px;background:#14243d;border:1px solid #2b4669;color:#e8f3ff;font-size:11.5px;line-height:1.35
}
.person-scene {height:145px;position:relative;width:150px;margin:2px auto 0}
.status-glow {position:absolute;width:84px;height:84px;border-radius:50%;left:33px;top:30px;background:rgba(49,207,255,.06);filter:blur(16px)}
.person-head {position:absolute;width:48px;height:50px;left:51px;top:10px;background:#e4ad82;border-radius:48% 48% 45% 45%;z-index:4}
.person-head.hair-a {box-shadow:inset 0 13px 0 #30231f}
.person-head.hair-b {box-shadow:inset 0 16px 0 #68452f}
.person-head.hair-c {box-shadow:inset 8px 10px 0 #1c1a20}
.person-head.hair-d {box-shadow:inset -8px 12px 0 #4c2f26}
.agent-icon {position:absolute;bottom:2px;left:50%;transform:translateX(-50%);font-size:15px}
.person-body {position:absolute;width:80px;height:63px;left:35px;top:50px;border-radius:34px 34px 12px 12px;z-index:2}
.shirt-cyan {background:linear-gradient(#168db6,#0c5c83)}
.shirt-purple {background:linear-gradient(#7650dc,#4430a3)}
.shirt-teal {background:linear-gradient(#22a58f,#116759)}
.shirt-blue {background:linear-gradient(#497fe3,#28499b)}
.person-arm {position:absolute;width:15px;height:45px;background:#dca779;top:65px;z-index:1;border-radius:8px}
.arm-left {left:29px;transform:rotate(20deg)} .arm-right {right:29px;transform:rotate(-20deg)}
.desk-top {position:absolute;left:9px;right:9px;height:10px;bottom:10px;border-radius:10px;background:linear-gradient(#3a5172,#22344e);z-index:5}
.laptop {position:absolute;width:76px;height:49px;left:37px;bottom:18px;border:2px solid #627ead;border-radius:7px;background:linear-gradient(150deg,#111c33,#070e1c);z-index:6;display:flex;align-items:center;justify-content:center;font-size:17px;box-shadow:0 0 12px rgba(84,142,255,.20)}
.agent-name {font-size:17px;color:#f2f7ff;font-weight:800;margin-top:2px}
.agent-role {font-size:11.5px;line-height:1.55;color:#99abc1;min-height:52px;margin:7px auto 10px;max-width:220px}
.agent-footer {display:flex;justify-content:center;gap:7px;align-items:center;font-size:10.5px}
.agent-code {font-family:monospace;color:#c8d7e9}
.agent-status {display:inline-flex;align-items:center;gap:5px;border:1px solid #d29b2b;border-radius:999px;padding:4px 8px;color:#dce9f7}
.agent-status i {width:6px;height:6px;border-radius:50%;background:#e0a62f}
.status-running {transform:translateY(-2px);box-shadow:0 0 22px rgba(47,201,255,.10);border-color:rgba(49,204,255,.48)}
.status-running .agent-bubble {background:#0d3041;border-color:#31ccff}.status-running .agent-status {border-color:#31ccff}.status-running .agent-status i {background:#31ccff;box-shadow:0 0 7px #31ccff}
.status-done .agent-bubble {background:#0d3029;border-color:#27d39c}.status-done .agent-status {border-color:#27d39c}.status-done .agent-status i {background:#27d39c}
.status-skipped {opacity:.45;filter:saturate(.5)}.status-skipped .agent-status {border-color:#728196}.status-skipped .agent-status i {background:#728196}
.status-error .agent-bubble {background:#3b1620;border-color:#ff6577}.status-error .agent-status {border-color:#ff6577}.status-error .agent-status i {background:#ff6577}


/* ---------- REHEARSAL MEDIA ---------- */
.media-section {
    margin-top:18px;
    padding:18px;
    border:1px solid rgba(75,211,255,.16);
    border-radius:18px;
    background:rgba(8,22,41,.72);
}
.media-section-title {color:#eaf4ff;font-size:16px;font-weight:800;margin-bottom:4px}
.media-section-text {color:#8fa7c1;font-size:12.5px;margin-bottom:12px;line-height:1.55}
.media-processing {
    background:rgba(11,48,65,.55);
    border:1px solid rgba(49,204,255,.35);
    color:#bfeeff;
    border-radius:16px;
    padding:18px;
    margin:8px 0;
}

/* ---------- REPORT ---------- */
.empty-report {min-height:460px;display:flex;flex-direction:column;justify-content:center;align-items:center;text-align:center;color:#9fb1c7}
.empty-icon {font-size:42px;margin-bottom:12px;opacity:.8}.empty-report h2 {color:#eaf3ff;font-size:25px;margin-bottom:8px}
.report-header {display:flex;align-items:center;justify-content:space-between;gap:20px;margin-bottom:20px}
.report-label {color:#55dfff;font-size:13px;font-weight:800;letter-spacing:1.2px;margin-bottom:7px}.report-header h2 {color:#f4f8ff;font-size:25px;margin:0}
.completed-badge {padding:8px 13px;border-radius:999px;background:rgba(38,211,148,.12);border:1px solid rgba(38,211,148,.25);color:#72f2bb;font-size:13px;font-weight:700}
.assessment-card {background:rgba(16,30,53,.85);border:1px solid rgba(82,192,255,.15);border-radius:17px;padding:20px;margin-bottom:20px}.assessment-card p {color:#c8d6e7;line-height:1.7;font-size:15px}
.section-title {color:#eaf3ff;font-size:17px;font-weight:800;margin-bottom:12px}.report-section {margin-top:23px}
.report-item {color:#bfcde0;background:rgba(12,25,45,.75);border:1px solid rgba(89,174,231,.10);border-radius:12px;padding:11px 14px;margin-bottom:8px;line-height:1.5}.muted {color:#7f92aa}
.metrics-grid {display:grid;grid-template-columns:repeat(auto-fit,minmax(120px,1fr));gap:10px;margin:18px 0}.metric-card {background:rgba(15,28,50,.85);border:1px solid rgba(84,196,255,.14);border-radius:14px;padding:14px}.metric-name {color:#8fa7c1;font-size:12px;margin-bottom:5px}.metric-value {color:#e8f5ff;font-size:18px;font-weight:800}
.error-card {background:rgba(100,24,35,.25);border:1px solid rgba(255,92,110,.35);border-radius:17px;padding:25px;color:#ffb9c2;margin-top:20px}

@media(max-width:1120px){.agents-grid{grid-template-columns:repeat(3,minmax(0,1fr))}}
@media(max-width:850px){.agents-grid{grid-template-columns:repeat(2,minmax(0,1fr))}.gradio-container{padding:10px!important}#talaqah-hero{padding:30px 20px}.team-board{padding:18px}.talaqah-card,#talaqah-report{padding:20px;min-height:auto}.report-header{flex-direction:column;align-items:flex-start}}
@media(max-width:520px){.agents-grid{grid-template-columns:1fr}.agent-card{min-height:305px}.team-board h2{font-size:23px}}
"""


with gr.Blocks(title="Talaqah | Your AI Presentation Coach", css=custom_css, theme=gr.themes.Base()) as talaqah_demo:
    gr.HTML("""<div id='talaqah-hero'><h1>🎤 طلاقة | Talaqah</h1><div class='subtitle'>Your AI Presentation Coach</div><div class='tagline'>Speak clearly. Present confidently. Improve continuously.</div><div class='hero-line'></div></div>""")
    with gr.Column(elem_classes="talaqah-card"):
        gr.HTML("""<div class='card-title'>✨ Start your coaching session</div><div class='card-text'>Add your topic and request. Upload any combination of transcript, presentation, audio, and video.</div>""")
        with gr.Row():
            topic_box = gr.Textbox(label="Presentation topic", placeholder="Example: Building Confidence in Public Speaking", scale=2)
            duration_box = gr.Number(label="Presentation duration (minutes)", value=0, minimum=0, step=0.1, scale=1, info="Use 0 for automatic audio/video duration. Enter the measured duration for transcript-only analysis.")
        request_box = gr.Textbox(label="What would you like Talaqah to analyze?", value="Analyze my slides and content, evaluate my delivery from the audio/video including pace, filler words and pauses, and give me a complete confidence improvement plan.", lines=4, elem_id="talaqah-input")
        gr.HTML("""<div class='media-section'><div class='media-section-title'>✅ Complete specialist team enabled</div><div class='media-section-text'>The capstone demo always runs Content & Slides + Delivery + Confidence.</div></div>""")
        with gr.Row():
            transcript_upload = gr.File(label="Rehearsal transcript (.txt)", file_types=[".txt"], type="filepath")
            presentation_upload = gr.File(label="Presentation (.pptx / .pdf)", file_types=[".pptx", ".pdf"], type="filepath")
        gr.HTML("""<div class='media-section'><div class='media-section-title'>🎙️ Real audio and video rehearsal</div><div class='media-section-text'>Audio is analyzed even when a transcript is present. Video adds visible-delivery observations without sensitive-trait or emotion inference. Duration is measured automatically when possible.</div></div>""")
        with gr.Row():
            audio_recording = gr.Audio(label="🎙️ Voice rehearsal", sources=["microphone", "upload"], type="filepath")
            video_recording = gr.Video(label="📹 Camera rehearsal", sources=["webcam", "upload"], format="mp4", include_audio=True)
        run_button = gr.Button("🚀 Start Talaqah Team", variant="primary", elem_id="talaqah-run")
    board_output = gr.HTML(render_agent_board(new_board_state()))
    with gr.Column(elem_id="talaqah-report"):
        gr.HTML("""<div class='card-title'>🎯 Your AI Coaching</div><div class='card-text'>The report is produced from the real multi-agent workflow and its tool evidence.</div>""")
        report_output = gr.HTML(render_report(None))
        run_state = gr.State(value=None)
        with gr.Row():
            approve_button = gr.Button("✅ Approve & Publish", visible=False, variant="primary")
            reject_button = gr.Button("❌ Reject", visible=False, variant="stop")
    run_button.click(
        fn=run_talaqah_live,
        inputs=[topic_box, request_box, transcript_upload, audio_recording, video_recording, presentation_upload, duration_box],
        outputs=[board_output, report_output, run_state, approve_button, reject_button],
        show_progress="minimal",
        concurrency_limit=1,
    )
    approve_button.click(fn=approve_talaqah, inputs=[run_state], outputs=[board_output, report_output, run_state, approve_button, reject_button], show_progress="minimal", concurrency_limit=1)
    reject_button.click(fn=reject_talaqah, inputs=[run_state], outputs=[board_output, report_output, run_state, approve_button, reject_button], show_progress="minimal", concurrency_limit=1)

print("✅ Talaqah UI ready: transcript + slides + real audio + real video")
print("✅ Multi-worker routing, parallel specialists, live board, LangSmith, and HITL connected")
talaqah_demo.queue(default_concurrency_limit=1, max_size=10).launch(share=True, debug=False)


/tmp/ipykernel_546/3119822549.py:441: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Talaqah | Your AI Presentation Coach", css=custom_css, theme=gr.themes.Base()) as talaqah_demo:


✅ Talaqah UI ready: transcript + slides + real audio + real video
✅ Multi-worker routing, parallel specialists, live board, LangSmith, and HITL connected
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://635e038c95f5f87650.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 17. Required write-up — one paragraph per rubric section

### 1. Agent fundamentals
Talaqah uses three LangChain specialist agents and real tools for RAG retrieval, PPTX/PDF analysis, transcript metrics, microphone/uploaded audio analysis, webcam/uploaded video analysis, and PowerPoint creation. Audio and video are not decorative UI inputs: their file paths enter the LangGraph workflow and the Delivery Agent must call `analyze_audio_file` and/or `analyze_video_file`. Pydantic validates the multi-destination `RoutePlan`, `CoachingReport`, and `ReportEvaluation` objects.

### 2. Multi-agent and routing architecture
The project implements Track A with a structured-output supervisor and three specialists: Content & Slides, Delivery, and Confidence. The LLM can select multiple destinations; attached presentations and rehearsal media add safe input-aware routing guarantees. The selected agents are scheduled as Functional API tasks before their results are awaited, creating real fan-out/fan-in execution. The full-team capstone demonstration visibly executes all three workers and then combines their evidence.

### 3. RAG pipeline
The presentation guides are loaded, split, embedded with Sentence Transformers, stored in FAISS, and retrieved semantically with filenames and similarity scores. Each specialist must call the retrieval tool so its recommendations are grounded in the supplied knowledge base rather than generic advice.

### 4. Context and state management
Short-term state uses `InMemorySaver` and `thread_id`; the same-thread test increases `turn_count`. Long-term learner facts use a separate `InMemoryStore`, and the cross-thread test retrieves the same saved profile for the same user from a different thread.

### 5. Human-in-the-loop
The main workflow uses `interrupt()` immediately before publication. The UI and evidence run pause with the final draft and evaluator score; `Command(resume={"action": "approve"})` continues the same thread. The report is written only after approval, while rejection publishes nothing.

### 6. Functional API and error handling
The workflow uses LangGraph `@task` and `@entrypoint`, parallel worker futures, `RetryPolicy`, a shared rate limiter, and media-specific retry handling for transient 429/503 errors. Missing transcript-only duration triggers a user-fixable interrupt, while audio/video duration is measured with FFprobe. File type/path validation and safe ZIP extraction produce explicit errors.

### 7. Workflow pattern
The named Evaluator–Optimizer pattern evaluates the combined multi-agent report, revises it using every issue, and evaluates it again. Both evaluation dictionaries are printed, while measured media/transcript values and source filenames are preserved during optimization.

### 8. LangSmith observability
Tracing uses both the exact course variable `LANGCHAIN_TRACING_V2` and current LangSmith variables. The trace-inspection cell verifies receipt of real runs and records the slowest inspected step, duration, and traced error count.


## 18. Production-readiness notes

- Secrets are read only from Colab Secrets and never hard-coded or committed.
- ZIP extraction rejects path traversal; uploaded paths and supported file formats are validated.
- FFmpeg/FFprobe and Pydub provide real duration, pause, silence, and signal measurements.
- Gemini provides qualitative speech/visual observations separately from deterministic numeric metrics.
- Audio is still analyzed when a transcript is uploaded; video is still analyzed when other inputs are present.
- The visual analysis describes observable presentation behavior only and does not infer emotion or sensitive traits.
- A shared rate limiter and retries reduce Gemini free-tier 429 failures; the UI prevents simultaneous duplicate runs.
- `InMemorySaver` and `InMemoryStore` are appropriate for the reproducible capstone; production should use durable stores.
- The approved artifact is saved only after human review.
- For submission: restart the runtime, Run all, upload the knowledge ZIP when prompted, keep successful outputs, verify every check is `PASS`, and confirm LangSmith traces.

**SDAIA Academy GitHub:** https://github.com/SDAIAAcademy
